# [실습] 회귀와 분류: 선형·로지스틱 회귀

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🤖 [실습] 선형 회귀와 로지스틱 회귀 — 예측의 첫걸음

## — 오늘 배운 것으로, 진짜 비즈니스 데이터의 질문에 답합니다

개념 노트북에서 지도학습 루프(**나누고 → 배우고 → 맞히고 → 정직하게 채점**)를 익혔습니다. 이제 그 루프를 **실제 서비스에서 수집된 데이터**에 적용합니다. 이번에는 문제가 주어지고, 코드는 여러분이 씁니다 — 막히면 힌트를, 그래도 막히면 예시 답안을 펼치세요.

> 🤖 **오늘의 AI 규칙 — 맨손 단계:** 코드는 손으로 씁니다. AI에게는 개념 *설명*만 요청하세요. (자세한 이유는 개념 노트북 Part 0)

## 📋 오늘의 미션

온라인 쇼핑몰의 마케팅팀이 물었습니다.

> "우리 사이트에 들어온 방문 세션 중 실제 구매로 이어지는 건 일부뿐입니다. **세션의 행동 데이터로, 구매로 이어질지 예측할 수 있을까요?**"

그리고 공공자전거 운영팀이 물었습니다.

> "날씨 정보로 **하루 자전거 대여량이 얼마나 될지** 미리 알 수 있을까요?"

첫 질문은 **분류**, 둘째 질문은 **회귀**입니다 — 오늘 배운 두 도구가 정확히 이 두 질문에 대응합니다.

| 문제 | 내용                            | 확인하는 힘                 |
| ---- | ------------------------------- | --------------------------- |
| 1    | 데이터 첫 대면 — 구조·타겟 확인 | 회귀/분류 판별              |
| 2    | 분류: 베이스라인부터            | "찍기"의 점수 재기          |
| 3    | 분류: 로지스틱 회귀로 예측      | 루프 완주 + 베이스라인 대비 |
| 4    | 회귀: 자전거 대여량 예측        | 같은 루프, 다른 문제 유형   |
| 5    | 첫 모델 카드 완성               | 기록하는 습관 (**제출물**)  |

> 📌 **오늘부터 같은 데이터를 계속 씁니다.** 이 쇼핑 세션 데이터는 앞으로 여덟 순서 동안 여러분과 함께 갑니다. 순서마다 도구가 하나씩 늘고, 그 도구로 **오늘 낸 답을 다시 검증**합니다. 그래서 저장소가 쌓이면 그것이 하나의 완결된 분석 포트폴리오가 됩니다 — 오늘 만드는 모델 카드 v1이 그 첫 장입니다.

> ⚠️ **오늘 일부러 남겨 두는 것 두 가지.** ① 18개 열 중 **수치형 10개만** 씁니다(문자 열을 넣는 법은 아직 안 배웠습니다). ② 지표는 **정확도만** 봅니다. 둘 다 지금은 한계지만, **그 한계를 모델 카드에 적어 두는 것**까지가 오늘의 과제입니다. 뒤 순서에서 하나씩 풀립니다.

# ⚙️ 데이터 준비

두 데이터 모두 **UCI 머신러닝 저장소**(UC Irvine Machine Learning Repository)의 공개 데이터입니다 — 수업용 장난감이 아니라 실제 서비스에서 수집된 기록입니다.

| 데이터                                   | 내용                                                                                             | 출처·라이선스                                                                                               |
| ---------------------------------------- | ------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------- |
| **Online Shoppers Purchasing Intention** | 온라인 쇼핑몰 방문 세션 12,330건 — 페이지 방문·체류·이탈률 등 18개 열, 타겟 `Revenue`(구매 여부) | [UCI 468](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset) · CC BY 4.0 |
| **Bike Sharing**                         | 미국 워싱턴 D.C. 공공자전거의 2년치 일별 대여 기록 731일 — 날씨·계절 정보, 타겟 `cnt`(대여량)    | [UCI 275](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) · CC BY 4.0                         |

[C1] 셀은 **최초 실행 시에만** 인터넷에서 내려받아 `data/` 폴더에 저장합니다. 이후에는 저장된 파일을 읽으므로 오프라인에서도 동작합니다.

▶ 실행: [C1]

In [1]:
# ─────────────────────────────────────────────
# [C1] ◀ ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))
bikes = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    "bike_sharing.zip", "day.csv"))

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"자전거 대여 데이터: {bikes.shape[0]:,}행 × {bikes.shape[1]}열")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
내려받는 중… bike_sharing.zip
쇼핑 세션 데이터: 12,330행 × 18열
자전거 대여 데이터: 731행 × 16열

→ 준비 완료. 이제 여러분 차례입니다.


# 문제 1. 데이터 첫 대면 — 이 문제는 회귀인가, 분류인가

실무 분석의 첫 단계는 언제나 "데이터가 어떻게 생겼는지"부터입니다. 그리고 그중 가장 먼저 볼 것은 **타겟 열**입니다 — 타겟에 들어 있는 값의 *종류*가 회귀냐 분류냐를 결정하고, 그 판단이 이후 모든 선택(모델·지표·분할 방법)을 갈라놓기 때문입니다.

타겟의 **분포**도 함께 봐야 합니다. 두 값 중 한쪽이 훨씬 많다면 그 사실 자체가 다음 문제의 함정을 만듭니다.

```
[문제 1]
1) shoppers의 앞 5행을 출력해 어떤 열들이 있는지 훑어보세요.
2) 타겟 열 `Revenue`에 어떤 값들이 들어 있는지 확인하세요.
   → 이 문제는 회귀 문제인가요, 분류 문제인가요? 근거는?
3) 전체 세션 중 구매(Revenue=True)로 이어진 비율을 계산하세요.
```

> 🤔 **예상해 볼까요?** 온라인 쇼핑몰에 들어온 방문 세션 100건 중, 실제 **구매까지 이어지는 건 몇 건**일 것 같습니까? 절반쯤일까요, 아니면 훨씬 적을까요? 숫자를 먼저 적어 두고 확인해 보세요.

▶ 실행: [C2]

In [13]:
# [C2] ◀ 문제 1. 데이터 첫 대면 — 이 문제는 회귀인가, 분류인가
# ⌨️ 문제 1 — 데이터 구조·타겟·구매 전환율 확인

# 여기에 코드를 작성하세요
# 1)
shoppers_5 = shoppers.head(5)
print(shoppers_5)
# 2)
target_Revenue = shoppers.Revenue.value_counts()
print(target_Revenue) #분류문제이다. True가 False로 나눠져있때문이다.
# 3)
norm = shoppers.Revenue.value_counts(normalize=True)
print(norm)

   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               1                 0.000000   
1                     0.0               2                64.000000   
2                     0.0               1                 0.000000   
3                     0.0               2                 2.666667   
4                     0.0              10               627.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0         0.20       0.20         0.0         0.0   Feb                 1   
1         0.00       0.10         0.0         0.0   Feb   

<details>
<summary>(클릭) 💡 힌트</summary>

- 앞 몇 행은 `.head()`, 열 정보는 `.info()`
- 타겟에 어떤 값이 몇 개씩 있는지는 `.value_counts()`
- 비율은 `.value_counts(normalize=True)` 또는 불리언 열의 `.mean()`

</details>

> 🎯 **[C2] 여기서 챙길 것:** 전환율 15.5% — 구매(True)가 훨씬 적은 데이터입니다. 이 사실이 바로 다음 문제에서 **정확도의 착시**를 만들어냅니다.

# 문제 2. 분류 — 베이스라인부터 세운다

개념 노트북 Part 5의 규칙: **성능은 항상 베이스라인 대비로 말한다.** 모델을 만들기 전에, "아무 패턴도 안 배우고 무조건 다수 클래스(비구매)로 찍는" 베이스라인의 정확도부터 잽시다.

이번 문제에서는 18개 열 중 **수치형 10개만** 씁니다 — `Month`·`VisitorType` 같은 문자(범주형) 열을 모델에 넣는 법은 아직 배우지 않았기 때문입니다(피처 다루기 시간에 배웁니다). 열 목록은 [C3]에 준비해 두었습니다.

```
[문제 2]
1) X(수치형 10개 열)와 y(Revenue를 0/1 정수로)를 만드세요.
2) train_test_split으로 8:2 분리하세요 — 분류이므로 stratify를 잊지 마세요.
3) DummyClassifier(strategy="most_frequent")를 학습시키고 테스트 정확도를 출력하세요.
```

> 🤔 **예상해 볼까요?** 문제 1에서 본 전환율(15.5%)을 기억하세요 — 무조건 "비구매"로만 찍는 베이스라인의 정확도는 몇쯤 나올까요?

▶ 실행: [C3]

In [14]:
# [C3] ◀ 문제 2. 분류 — 베이스라인부터 세운다
# ⌨️ 문제 2 — 베이스라인(Dummy) 정확도
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

# 여기에 코드를 작성하세요 (X, y 준비 → 분리 → Dummy 학습 → 테스트 정확도)
# 1)
X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)
# 2)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE,stratify=y)
# 3)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print(dummy.score(X_test, y_test))

0.8450932684509327


<details>
<summary>(클릭) 💡 힌트</summary>

- `X = shoppers[NUM_COLS]`, `y = shoppers["Revenue"].astype(int)`
- 분리: `train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)`
- Dummy도 여느 모델과 똑같이 `fit` → `score`

</details>

> 🎯 **[C3] 예상과 맞았나요?** 베이스라인 정확도는 약 **0.845**입니다. 데이터를 한 줄도 배우지 않고 "전부 비구매"라고 찍기만 해도 84.5%를 맞힙니다. **"우리 모델 정확도 84%!"라는 보고가 얼마나 공허할 수 있는지** — 개념 노트북에서 배운 베이스라인 규칙이 실데이터에서 이렇게 실감됩니다.

# 문제 3. 분류 — 로지스틱 회귀로 베이스라인을 넘어라

이제 진짜 모델 차례입니다. 마케팅팀의 질문에 처음으로 답해 봅시다.

```
[문제 3]
1) 학습 데이터로 스케일링 기준을 잡아(fit) 학습/테스트 데이터를 변환하세요.
2) LogisticRegression(max_iter=5000)을 학습시키세요.
3) 다음 세 값을 출력하고 비교하세요.
   - 베이스라인 정확도 (문제 2)
   - 로지스틱 회귀의 테스트 정확도
   - 로지스틱 회귀의 학습 정확도
   → 베이스라인보다 얼마나 나은가요? 과적합 신호가 있나요?
```

> 🤔 **예상해 볼까요?** 베이스라인은 0.845였습니다. 진짜 모델이 데이터를 학습하면 정확도가 어디까지 오를까요? 0.95쯤일까요? 그리고 **학습 정확도와 테스트 정확도** 중 어느 쪽이 높게 나올 것 같습니까?

▶ 실행: [C4]

In [16]:
# [C4] ◀ 문제 3. 분류 — 로지스틱 회귀로 베이스라인을 넘어라
# ⌨️ 문제 3 — 로지스틱 회귀 학습·평가 (베이스라인과 비교)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 여기에 코드를 작성하세요 (스케일링 → 학습 → 학습/테스트 정확도 비교)
# 1)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  
X_test_s = scaler.transform(X_test)         
# 2)
logreg = LogisticRegression(max_iter=5000)
logreg.fit(X_train_s, y_train)
# 3)
baseline_acc = dummy.score(X_test, y_test)
train_acc = logreg.score(X_train_s, y_train)
test_acc = logreg.score(X_test_s, y_test)

print(f"베이스라인: {baseline_acc:.3f}")
print(f"로지스틱회귀 테스트 정확도: {test_acc:.3f}")
print(f"로지스틱회귀 학습 정확도: {train_acc:.3f}")

베이스라인: 0.845
로지스틱회귀 테스트 정확도: 0.880
로지스틱회귀 학습 정확도: 0.884


<details>
<summary>(클릭) 💡 힌트</summary>

- 스케일링은 개념 노트북 Part 4와 같은 패턴: `fit_transform`은 학습 데이터에만, 테스트는 `transform`만
- 학습 정확도 = `model.score(X_train_s, y_train)`, 테스트 정확도 = `model.score(X_test_s, y_test)`

</details>

> 🎯 **[C4] 읽는 법:** "+0.034"가 작아 보이나요? 12,330 세션 규모에서는 **수백 건의 구매를 더 정확히 겨냥**할 수 있다는 뜻입니다. 그리고 아직 우리는 문자(범주형) 열 8개를 쓰지도 않았습니다 — 더 끌어올릴 여지는 **피처 다루기 시간**에 만납니다.
>
> 한 가지 더: 전환율 15.5%짜리 불균형 데이터에서 정확도만 보는 게 영 찜찜하지 않았나요? 그 찜찜함이 정확합니다 — **분류 지표·불균형 시간**에 정면으로 다룹니다.

# 문제 4. 회귀 — 하루 대여량을 예측하라

같은 루프를 이번엔 **회귀** 문제에 적용합니다. 자전거 운영팀의 질문: 날씨로 하루 대여량을 예측할 수 있을까?

```
[문제 4]
1) bikes에서 피처 4개(temp, atemp, hum, windspeed)와 타겟 cnt로 X, y를 만드세요.
2) 8:2로 분리하세요 (회귀에는 stratify가 필요 없습니다 — 왜일까요?).
3) DummyRegressor(평균 예측)와 LinearRegression을 각각 학습시키고,
   테스트 R²를 비교하세요.

⚠️ casual, registered 열은 절대 쓰면 안 됩니다 — 이유를 생각해 보세요. (답안에서 확인)
```

> 🤔 **예상해 볼까요?** 회귀의 베이스라인은 "언제나 전체 평균을 답한다"입니다. 그 모델의 **R²는 얼마**가 나올까요? 0.5쯤일까요, 0일까요, 아니면 **음수**도 가능할까요?

▶ 실행: [C5]

In [17]:
# [C5] ◀ 문제 4. 회귀 — 하루 대여량을 예측하라
# ⌨️ 문제 4 — 자전거 대여량 회귀 (Dummy → LinearRegression)
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

FEATURES = ["temp", "atemp", "hum", "windspeed"]

# 여기에 코드를 작성하세요 (X, y → 분리 → Dummy R² vs 선형회귀 R²)
# 1)
X = bikes[FEATURES]
y = bikes["cnt"]
# 2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)  
#cnt가 연속형이라 필요없음
# 3)
dummy_reg = DummyRegressor(strategy="mean")
dummy_reg.fit(X_train, y_train)
dummy_r2 = dummy_reg.score(X_test, y_test)

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
train_r2 = lin_reg.score(X_train, y_train)
test_r2 = lin_reg.score(X_test, y_test)

print(f"베이스라인 R²: {dummy_r2:.3f}")
print(f"학습 R²: {train_r2:.3f}")
print(f"테스트 R²: {test_r2:.3f}")



베이스라인 R²: -0.020
학습 R²: 0.450
테스트 R²: 0.499


<details>
<summary>(클릭) 💡 힌트</summary>

- `X = bikes[FEATURES]`, `y = bikes["cnt"]`
- 회귀의 `score`는 R²를 돌려줍니다 — Dummy(평균 예측)의 R²는 0 근처가 정상
- `stratify`는 **클래스 비율**을 지키는 장치라서, 연속 숫자 타겟(회귀)에는 개념 자체가 성립하지 않습니다

</details>

> 🎯 **[C5] 예상과 맞았나요?** 세 가지가 한꺼번에 드러납니다.
>
> ① **베이스라인 R²가 음수(−0.020)입니다.** R²는 "평균만 답하는 모델보다 얼마나 나은가"를 재는 값이라 그 모델 자신은 0 근처가 되고, 분할에 따라 살짝 음수도 나옵니다. **정확도와 달리 R²는 하한이 0이 아닙니다.**
>
> ② **테스트 R² 0.499가 학습 R² 0.450보다 높습니다.** 과적합의 _반대_ 현상입니다. 731일이라는 작은 데이터에서 8:2로 한 번 나눴을 뿐이니, 이건 실력이 아니라 **분할 운**입니다. 이 흔들림을 제대로 재는 법(교차 검증)은 뒤 순서에서 배웁니다.
>
> ③ **`casual`·`registered`를 넣으면 R²가 정확히 1.0000이 됩니다.** 두 열의 합이 곧 타겟이니까요. 완벽한 점수는 축하할 일이 아니라 **의심할 일**입니다 — 실무에서 R²가 0.99를 넘으면 가장 먼저 누수를 찾습니다.

# 문제 5. 첫 모델 카드 완성 — 오늘의 제출물

두 문제의 결과를 **모델 카드 v1**으로 기록합니다. 개념 노트북에서 배운 규칙 그대로 — 베이스라인 대비, 학습-테스트 비교, 그리고 한계까지.

```
[문제 5]
1) [C6]에서 두 모델(쇼핑 분류·자전거 회귀)의 결과를 하나의 표로 만드세요.
   (열: 문제, 유형, 모델, 주요 지표, 베이스라인, 학습, 테스트)
2) 아래 모델 카드 템플릿의 빈칸을 여러분의 숫자와 문장으로 채우세요.
```

▶ 실행: [C6]

In [18]:
# [C6] ◀ 문제 5. 첫 모델 카드 완성 — 오늘의 제출물
# ⌨️ 문제 5 — 두 모델을 하나의 표로

# 여기에 코드를 작성하세요 (pd.DataFrame으로 모델 카드 표 만들기)
model_card = pd.DataFrame([
    {
        "문제": "구매 전환 예측", "유형": "분류", "모델": "LogisticRegression",
        "주요 지표": "정확도",
        "베이스라인": round(baseline_acc, 3),
        "학습": round(train_acc, 3),
        "테스트": round(test_acc, 3),
    },
    {
        "문제": "자전거 대여량 예측", "유형": "회귀", "모델": "LinearRegression",
        "주요 지표": "R²",
        "베이스라인": round(dummy_r2, 3),
        "학습": round(train_r2, 3),
        "테스트": round(test_r2, 3),
    },
])
model_card


,문제,유형,모델,주요 지표,베이스라인,학습,테스트
0,구매 전환 예측,분류,LogisticRegression,정확도,0.845,0.884,0.880
1,자전거 대여량 예측,회귀,LinearRegression,R²,-0.020,0.450,0.499


**모델 카드 v1 템플릿** — 아래 빈칸을 채워 이 셀에 완성하세요 (셀을 더블클릭해 편집).

```markdown
## 모델 카드 v1 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330 세션, 수치형 10개 열만 사용)
- 문제 유형: 분류
- 모델: 베이스라인(Dummy) → LogisticRegression
- 평가 지표 & 이유: 이진분류, 정확도만 다루기로 한계를 정했기 때문
- 성능: 베이스라인 0.845 → 내 모델 0.879 / 학습 0.884 vs 테스트 0.88
- 과적합 진단: 과적합 차이가 0.004이니 거의 없는 수준 -> 과적합 없음
- 한계 & 다음 단계: 불균형인데 정확도만 봄 -> 정밀도 학습?

## 모델 카드 v1 — 자전거 대여량 예측

- 데이터: UCI Bike Sharing (731일, 날씨 피처 4개)
- 문제 유형: 회귀
- 모델: 베이스라인(Dummy) → LinearRegression
- 평가 지표 & 이유: R², 연속값(대여량)을 맞히는 회귀 문제이기 때문
- 성능: 베이스라인 -0.020 → 내 모델 0.499 / 학습 0.45 vs 테스트 0.499
- 과적합 진단: 테스트가 오히려 학습보다 낮음 -> 과적합 아님
- 한계 & 다음 단계: 날씨 피처 4개만 사용, 다른 열 미사용
```

**제출:** 이 노트북(작성 셀 + 모델 카드 완성본)을 개인 공개 저장소 main에 커밋·푸시하세요.

**스스로 점검하는 기준**

| 축            | 기준                                                              |
| ------------- | ----------------------------------------------------------------- |
| 판별 정확성   | 두 문제가 각각 분류·회귀인 근거를 타겟 값의 종류로 설명했는가     |
| 비교의 완전성 | 두 모델 모두 **베이스라인 대비**로 성능을 적었는가                |
| 진단 정확성   | 학습 점수와 테스트 점수를 함께 보고 과적합 여부를 판정했는가      |
| 정직성        | 오늘 쓰지 못한 것(범주형 8개 열·정확도 외 지표)을 한계에 적었는가 |

> 🚀 **더 나아가기:** `PageValues` 하나만으로 로지스틱 회귀를 학습하면 정확도가 얼마나 나올까요? 피처 10개 vs 1개 — 직접 비교해 보세요. 의외의 결과가 기다립니다.

오늘 여러분은 배운 지 하루도 안 된 도구로 **진짜 비즈니스 질문 두 개에 첫 답**을 내놓았습니다 — 그리고 그 답을 베이스라인과 비교하며 *의심하는 법*까지 함께 썼습니다.

오늘도 한 걸음, 수고하셨습니다! 🎉
어제의 나보다 데이터를 다루는 손이 한 뼘 더 능숙해졌습니다. 다음 시간에 또 만나요.

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>